In [21]:
### Colour distribution for RGB image

## Material_classification was obtained manually by feeding the DET-Compass 370 classes

In [2]:
import json
import os
from tqdm import tqdm
from pycocotools.coco import COCO
from PIL import Image
import numpy as np
import pycocotools.mask as mask_util

In [15]:
MATERIAL_JSON = "./material_classification.json"

SUPPORT_COCO = "./full_30_support_set_with_masks.json"

IMAGES_PATH = "./images"

OUT_FOLDER = "./colour_outputs"
os.makedirs(OUT_FOLDER, exist_ok=True)

dataset_names = ["PIXray"]#("PIXray", "pidray", "CLCXray", "HiXray", "DvXray", "xray_fsod", "eds", "COMPASS-XP")


In [16]:
with open(MATERIAL_JSON, "r") as f:
    materials = json.load(f)

print("Loaded material groups:")
for k in materials.keys():
    print("-", k)

Loaded material groups:
- metal
- plastic
- textile
- paper
- glass_ceramic
- wood
- rubber
- electronic
- organic_food
- chemical
- mixed_other


In [19]:
def clustering(d_name, support, material):
    mean_colors = []

    total_available = 0
    total_used = 0

    for category in support.loadCats(support.getCatIds()):
        if category['name'] not in material[1]:
            continue

        category_id = category['id']

        img_ids = support.getImgIds(catIds=[category_id])
        img_info_list = support.loadImgs(img_ids)

        total_available += len(img_info_list)

        # Exclude images from the current dataset
        imgs_filtered = [img for img in img_info_list if d_name not in img['file_name']]

        total_used += len(imgs_filtered)

        for img_info in imgs_filtered:
            img_path = os.path.join(IMAGES_PATH, img_info['file_name'])

            image = Image.open(img_path).convert("RGB")
            image_np = np.array(image)

            ann_ids = support.getAnnIds(imgIds=[img_info['id']], catIds=[category_id])
            anns = support.loadAnns(ann_ids)

            for ann in anns:
                mask = mask_util.decode(ann['mask'])

                if np.sum(mask) == 0:
                    continue

                cropped_object = image_np * np.expand_dims(mask, axis=-1)

                mean_color = np.mean(cropped_object[mask > 0], axis=0)
                mean_colors.append(mean_color)

    print(f"\nFor material group '{material[0]}':")
    print(f"  Total candidate images: {total_available}")
    print(f"  Images actually used after exclusion: {total_used}")
                

    if mean_colors:
        avg_mean_color = np.mean(mean_colors, axis=0)
        return avg_mean_color.astype(int)
    else:
        # If no data found, return neutral gray as fallback
        return np.array([128, 128, 128])


In [20]:
support = COCO(SUPPORT_COCO)
total_images = len(support.getImgIds())
print("Total images in support dataset:", total_images)

for d_name in tqdm(dataset_names):
    print(f"\nProcessing dataset: {d_name}")

    mat_colour = {}
    color_dict_per_cat = {}

    for mat in materials.items():
        material_name = mat[0]
        color_mat = clustering(d_name, support, mat)
        color_mat = color_mat.tolist()

        mat_colour[material_name] = color_mat

        # Assign same color to each category in that material
        for cat in mat[1]:
            color_dict_per_cat[cat] = color_mat

    # Save material → color mapping
    save_name = os.path.join(OUT_FOLDER, f"{d_name}_colour.json")
    with open(save_name, 'w') as out_file:
        json.dump(mat_colour, out_file, indent=4)
    print(f"Saved: {save_name}")

    # Save category → color mapping
    save_name = os.path.join(OUT_FOLDER, f"{d_name}_colour_per_cat.json")
    with open(save_name, 'w') as out_file:
        json.dump(color_dict_per_cat, out_file, indent=4)
    print(f"Saved: {save_name}")


loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Total images in support dataset: 2838


  0%|                                                                                                                                                                                          | 0/1 [00:00<?, ?it/s]


Processing dataset: PIXray

For material group 'metal':
  Total candidate images: 361
  Images actually used after exclusion: 361

For material group 'plastic':
  Total candidate images: 264
  Images actually used after exclusion: 264

For material group 'textile':
  Total candidate images: 51
  Images actually used after exclusion: 51

For material group 'paper':
  Total candidate images: 12
  Images actually used after exclusion: 12

For material group 'glass_ceramic':
  Total candidate images: 43
  Images actually used after exclusion: 43

For material group 'wood':
  Total candidate images: 16
  Images actually used after exclusion: 16

For material group 'rubber':
  Total candidate images: 0
  Images actually used after exclusion: 0

For material group 'electronic':
  Total candidate images: 127
  Images actually used after exclusion: 127

For material group 'organic_food':
  Total candidate images: 46
  Images actually used after exclusion: 46

For material group 'chemical':
  T

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:08<00:00,  8.87s/it]


For material group 'mixed_other':
  Total candidate images: 77
  Images actually used after exclusion: 77
Saved: ./colour_outputs/PIXray_colour.json
Saved: ./colour_outputs/PIXray_colour_per_cat.json


In [22]:
### web image processing

In [5]:
import os
import json
from tqdm import tqdm
from ultralytics import YOLO
from PIL import Image

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/haree/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
WEB_IMAGE_ROOT = "web_images/"
OUTPUT_JSON = "web_image_crops.json"

CONF_THRESHOLD = 0.4

In [7]:
model = YOLO("yolov8m.pt")  
print("YOLO model loaded!")

YOLO model loaded!


In [8]:
final_json = {"images": []}

for category in os.listdir(WEB_IMAGE_ROOT):
    cat_dir = os.path.join(WEB_IMAGE_ROOT, category)

    if not os.path.isdir(cat_dir):
        continue

    print("\nProcessing category:", category)

    for img_file in tqdm(os.listdir(cat_dir)):
        img_path = os.path.join(cat_dir, img_file)

        try:
            results = model(img_path)

            for r in results:
                boxes = r.boxes

                if len(boxes) == 0:
                    continue

                best_box = boxes[0]
                conf = float(best_box.conf)

                if conf < CONF_THRESHOLD:
                    continue

                x1, y1, x2, y2 = best_box.xyxy[0].tolist()

                final_json["images"].append({
                    "image_name": f"{category}/{img_file}",
                    "category": category,
                    "bbox_xyxy": [x1, y1, x2, y2]
                })

        except Exception as e:
            print("Error on:", img_file, e)



Processing category: violin


  0%|                                                                                                                                                                                          | 0/3 [00:00<?, ?it/s]


image 1/1 /Users/haree/X-Ray Project/Notebooks/web_images/violin/image2.jpeg: 640x480 1 vase, 127.0ms
Speed: 2.0ms preprocess, 127.0ms inference, 3.9ms postprocess per image at shape (1, 3, 640, 480)


 33%|███████████████████████████████████████████████████████████▎                                                                                                                      | 1/3 [00:00<00:00,  4.56it/s]


image 1/1 /Users/haree/X-Ray Project/Notebooks/web_images/violin/image1.jpg: 640x384 (no detections), 92.8ms
Speed: 0.6ms preprocess, 92.8ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

image 1/1 /Users/haree/X-Ray Project/Notebooks/web_images/violin/image3.jpeg: 640x480 1 motorcycle, 107.1ms
Speed: 0.6ms preprocess, 107.1ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 480)


...

 33%|███████████████████████████████████████████████████████████▎                                                                                                                      | 1/3 [00:00<00:00,  6.29it/s]


image 1/1 /Users/haree/X-Ray Project/Notebooks/web_images/knife/image2.jpg: 448x640 1 scissors, 102.3ms
Speed: 0.8ms preprocess, 102.3ms inference, 0.3ms postprocess per image at shape (1, 3, 448, 640)


 67%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                           | 2/3 [00:00<00:00,  7.79it/s]


image 1/1 /Users/haree/X-Ray Project/Notebooks/web_images/knife/image3.jpg: 448x640 1 knife, 102.7ms
Speed: 0.9ms preprocess, 102.7ms inference, 0.2ms postprocess per image at shape (1, 3, 448, 640)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  7.99it/s]


In [9]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(final_json, f, indent=4)

print("\nSaved annotations to:", OUTPUT_JSON)
print("Total detected objects:", len(final_json["images"]))



Saved annotations to: web_image_crops.json
Total detected objects: 6


In [10]:
### SAM2

In [11]:
import os
import json
import copy
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch
import pycocotools.mask as mask_util

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)


Using device: cpu


In [13]:
# JSON produced by YOLO detection
GT_JSON = "web_image_crops.json"

# Folder containing your original images
IMAGE_PATH = "web_images/"

# Output JSON with SAM masks
OUTPUT_JSON = GT_JSON.replace(".json", "_with_masks.json")


In [14]:
def single_mask_to_rle(mask):
    rle = mask_util.encode(
        np.array(mask[:, :, None], order="F", dtype="uint8")
    )[0]
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle


In [27]:
import hydra
from hydra import initialize, compose
import os


In [31]:
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from hydra import initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

device = "cpu"

# Absolute path to your SAM2 config folder
CONFIG_DIR = os.path.join(os.getcwd(), "segment-anything-2/sam2/configs/sam2.1")

def build_sam_model():

    sam2_checkpoint = "sam2.1_hiera_large.pt"
    model_cfg = "sam2.1_hiera_l.yaml"

    # --- FIX: clear previous hydra initialization ---
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()

    # Initialize Hydra with your local config directory
    initialize_config_dir(config_dir=CONFIG_DIR, version_base=None)

    sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
    sam2_predictor = SAM2ImagePredictor(sam2_model)

    return sam2_predictor


print("Loading SAM2 model...")
sam2_predictor = build_sam_model()
print("SAM2 model loaded!")


Loading SAM2 model...
SAM2 model loaded!


In [32]:
dataset = json.load(open(GT_JSON))
images_anots = dataset["images"]

print("Total images to process:", len(images_anots))


Total images to process: 6


In [33]:
images_anots_with_masks = []

for image_anot in tqdm(images_anots):

    img_path = os.path.join(IMAGE_PATH, image_anot["image_name"])

    try:
        image = Image.open(img_path)
    except:
        print("Could not open:", img_path)
        continue

    sam2_predictor.set_image(np.array(image.convert("RGB")))

    boxes_xyxy = np.array([image_anot["bbox_xyxy"]])

    masks, _, _ = sam2_predictor.predict(
        point_coords=None,
        point_labels=None,
        box=boxes_xyxy,
        multimask_output=False,
    )

    if masks.ndim == 4:
        masks = masks.squeeze(1)

    new_image_anot = copy.deepcopy(image_anot)
    new_image_anot["mask"] = single_mask_to_rle(masks[0])

    images_anots_with_masks.append(new_image_anot)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:13<00:00,  2.20s/it]


In [34]:
with open(OUTPUT_JSON, "w") as fp:
    json.dump(images_anots_with_masks, fp, indent=4, sort_keys=True)

print("\nSaved SAM masks to:", OUTPUT_JSON)



Saved SAM masks to: web_image_crops_with_masks.json


In [35]:
## style transfer

In [40]:
import os
import json
import numpy as np
from PIL import Image
import pycocotools.mask as mask_util
from tqdm import tqdm

# ----- INPUTS -----
GT_JSON = "web_image_crops_with_masks.json"   # list-based JSON
IMAGE_PATH = "web_images"                    # folder with RGB images
COLOUR_JSON = "PIXray_colour_per_cat.json"   # colour knowledge

# ----- OUTPUT -----
OUTPUT_DIR = "style_transfer_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load data
crops = json.load(open(GT_JSON))
category_avg_colors = json.load(open(COLOUR_JSON))

print(f"Loaded {len(crops)} masked crops")

for crop in tqdm(crops):

    img_path = os.path.join(IMAGE_PATH, crop["image_name"])

    if not os.path.exists(img_path):
        print(f"Skipping missing image: {img_path}")
        continue

    image = Image.open(img_path).convert("RGB")
    image_array = np.array(image)

    mask = mask_util.decode(crop["mask"])

    output = np.zeros_like(image_array)

    # Prefer super_category if present (RAXO design)
    if "super_category" in crop:
        key = crop["super_category"]
    else:
        key = crop["category"]

    if key not in category_avg_colors:
        print(f"No colour for category {key}, skipping")
        continue

    output[mask > 0] = category_avg_colors[key]

    output_image = Image.fromarray(output)
    output_image.save(os.path.join(OUTPUT_DIR, crop["image_name"]))

print("\n✅ Style transfer completed")
print(f"📁 Output folder: {OUTPUT_DIR}")


Loaded 6 masked crops


 50%|█████████████████████████████████████████████████████████████████████████████████████████                                                                                         | 3/6 [00:00<00:00, 22.16it/s]

No colour for category gun, skipping
No colour for category gun, skipping


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 29.51it/s]


✅ Style transfer completed
📁 Output folder: style_transfer_output
